В цьому домашньому завданні ми знову працюємо з даними з нашого змагання ["Bank Customer Churn Prediction (DLU Course)"](https://www.kaggle.com/t/7c080c5d8ec64364a93cf4e8f880b6a0).

Тут ми побудуємо рішення задачі класифікації з використанням алгоритмів бустингу: XGBoost та LightGBM, а також використаємо бібліотеку HyperOpt для оптимізації гіперпараметрів.

0. Зчитайте дані `train.csv` в змінну `raw_df` та скористайтесь наведеним кодом нижче аби розділити дані на трнувальні та валідаційні і розділити дані на ознаки з матириці Х та цільову змінну. Назви змінних `train_inputs, train_targets, train_inputs, train_targets` можна змінити на ті, які Вам зручно.

  Наведений скрипт - частина отриманого мною скрипта для обробки даних. Ми тут не викнуємо масштабування та обробку категоріальних змінних, бо хочемо це делегувати алгоритмам, які будемо використовувати. Якщо щось не розумієте в наведених скриптах, рекомендую розібратись: навичка читати код - важлива складова роботи в машинному навчанні.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from typing import Tuple, Dict, Any


def split_train_val(df: pd.DataFrame, target_col: str, test_size: float = 0.2, random_state: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Split the dataframe into training and validation sets.

    Args:
        df (pd.DataFrame): The raw dataframe.
        target_col (str): The target column for stratification.
        test_size (float): The proportion of the dataset to include in the validation split.
        random_state (int): Random state for reproducibility.

    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Training and validation dataframes.
    """
    train_df, val_df = train_test_split(df, test_size=test_size, random_state=random_state, stratify=df[target_col])
    return train_df, val_df


def separate_inputs_targets(df: pd.DataFrame, input_cols: list, target_col: str) -> Tuple[pd.DataFrame, pd.Series]:
    """
    Separate inputs and targets from the dataframe.

    Args:
        df (pd.DataFrame): The dataframe.
        input_cols (list): List of input columns.
        target_col (str): Target column.

    Returns:
        Tuple[pd.DataFrame, pd.Series]: DataFrame of inputs and Series of targets.
    """
    inputs = df[input_cols].copy()
    targets = df[target_col].copy()
    return inputs, targets

In [3]:
raw_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/train.csv')

target_col = 'Exited'
cols_to_drop = [target_col, 'id', 'CustomerId', 'Surname']
input_cols = [col for col in raw_df.columns if col not in cols_to_drop]

train_df, val_df = split_train_val(raw_df, target_col)

train_inputs, train_targets = separate_inputs_targets(train_df, input_cols, target_col)
val_inputs, val_targets = separate_inputs_targets(val_df, input_cols, target_col)

1. В тренувальному та валідаційному наборі перетворіть категоріальні ознаки на тип `category`. Можна це зробити двома способами:
 1. `df[col_name].astype('category')`, як було продемонстровано в лекції
 2. використовуючи метод `pd.Categorical(df[col_name])`

In [4]:
categorical_cols = train_inputs.select_dtypes(include=['object']).columns.tolist()

for col in categorical_cols:
    train_inputs[col] = train_inputs[col].astype('category')
    val_inputs[col] = val_inputs[col].astype('category')

print(f"Ознаки для навчання: {input_cols}")
print(f"Категоріальні колонки: {categorical_cols}")
print("\nТипи даних після перетворення:")
print(train_inputs.dtypes[categorical_cols])

Ознаки для навчання: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Категоріальні колонки: ['Geography', 'Gender']

Типи даних після перетворення:
Geography    category
Gender       category
dtype: object


2. Навчіть на отриманих даних модель `XGBoostClassifier`. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів XGBoostClassifier - тут https://xgboost.readthedocs.io/en/stable/parameter.html#global-config

  **Важливо:** зробіть такі налаштування `XGBoostClassifier` аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Можна також, якщо працюєте в Google Colab, увімкнути можливість використання GPU (`Runtime -> Change runtime type -> T4 GPU`) і встановити параметр `device='cuda'` в `XGBoostClassifier` для пришвидшення тренування бустинг моделі.
  
  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням DecisionTrees раніше. Чи вийшло покращити якість?

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    enable_categorical=True,
    tree_method='hist',
    random_state=42,
    use_label_encoder=False
)

xgb_clf.fit(train_inputs, train_targets)

def evaluate_model(model, inputs, targets, name=''):
    probs = model.predict_proba(inputs)[:, 1]
    auc = roc_auc_score(targets, probs)
    print(f"AUROC {name}: {auc:.4f}")
    return auc

print("Результати базової моделі XGBoost:")
train_auc = evaluate_model(xgb_clf, train_inputs, train_targets, "Train")
val_auc = evaluate_model(xgb_clf, val_inputs, val_targets, "Val")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:16:07] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Результати базової моделі XGBoost:
AUROC Train: 0.9690
AUROC Val: 0.9308


Є перенавчання. Бустинг показав вищу точність ніж дерева.

3. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `XGBoostClassifier` з лекції знайдіть оптимальні значення гіперпараметрів `XGBoostClassifier` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **20**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. В ній ми маємо задати loss - це може будь-яка метрика, але бажано використовувтаи ту, яка цільова в вашій задачі. Чим менший лосс - тим ліпша модель на думку hyperopt. Тож, тут нам треба задати loss - негативне значення AUROC. В лекції ми натомість використовували Accuracy.

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_clf` модель `XGBoostClassifier` з найкращими гіперпараметрами
    - навчіть модель `final_clf`
    - оцініть якість моделі `final_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (2) цього завдання?

In [6]:
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials

space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'max_depth': hp.quniform('max_depth', 3, 12, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'min_child_weight': hp.quniform('min_child_weight', 1, 10, 1),
    'gamma': hp.uniform('gamma', 0, 5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 10),
    'reg_lambda': hp.uniform('reg_lambda', 0, 10)
}

def objective(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['max_depth'] = int(params['max_depth'])
    params['min_child_weight'] = int(params['min_child_weight'])

    model = XGBClassifier(
        **params,
        enable_categorical=True,
        tree_method='hist',
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_inputs, train_targets)

    probs = model.predict_proba(val_inputs)[:, 1]
    auc = roc_auc_score(val_targets, probs)

    return {'loss': 1 - auc, 'status': STATUS_OK}

trials = Trials()
best_params = fmin(
    fn=objective,
    space=space,
    algo=tpe.suggest,
    max_evals=20,
    trials=trials,
    rstate=np.random.default_rng(42)
)

print("\nНайкращі параметри:")
final_params = best_params.copy()
final_params['n_estimators'] = int(final_params['n_estimators'])
final_params['max_depth'] = int(final_params['max_depth'])
print(final_params)

100%|██████████| 20/20 [00:09<00:00,  2.08trial/s, best loss: 0.06232114685506551]

Найкращі параметри:
{'colsample_bytree': np.float64(0.6896437006810046), 'gamma': np.float64(4.22980000436168), 'learning_rate': np.float64(0.03305486342407529), 'max_depth': 6, 'min_child_weight': np.float64(10.0), 'n_estimators': 300, 'reg_alpha': np.float64(0.5407787444418399), 'reg_lambda': np.float64(7.713808175958373), 'subsample': np.float64(0.8616492018371803)}


In [7]:
final_model_params = final_params.copy()
final_model_params['enable_categorical'] = True
final_model_params['tree_method'] = 'hist'
final_model_params['random_state'] = 42

final_clf = XGBClassifier(**final_model_params)

final_clf.fit(train_inputs, train_targets)

def get_auc(model, inputs, targets):
    probs = model.predict_proba(inputs)[:, 1]
    return roc_auc_score(targets, probs)

train_auc_final = get_auc(final_clf, train_inputs, train_targets)
val_auc_final = get_auc(final_clf, val_inputs, val_targets)

print(f"AUROC Train: {train_auc_final:.4f}")
print(f"AUROC Val:   {val_auc_final:.4f}")
print(f"Різниця:     {train_auc_final - val_auc_final:.4f}")

AUROC Train: 0.9414
AUROC Val:   0.9377
Різниця:     0.0037


Ця модель є найкращою з усіх, що були до цього часу.

4. Навчіть на наших даних модель LightGBM. Параметри алгоритму встановіть на свій розсуд, ми далі будемо їх тюнити. Рекомендую тренувати не дуже складну модель.

  Опис всіх конфігураційних параметрів LightGBM - тут https://lightgbm.readthedocs.io/en/latest/Parameters.html

  **Важливо:** зробіть такі налаштування LightGBM аби він самостійно обробляв незаповнені значення в даних і обробляв категоріальні колонки.

  Аби передати категоріальні колонки в LightGBM - необхідно виявити їх індекси і передати в параметрі `cat_feature=cat_feature_indexes`

  Після тренування моделі
  1. Виміряйте точність з допомогою AUROC на тренувальному та валідаційному наборах.
  2. Зробіть висновок про отриману модель: вона хороша/погана, чи є high bias/high variance?
  3. Порівняйте якість цієї моделі з тою, що ви отрмали з використанням XGBoostClassifier раніше. Чи вийшло покращити якість?

In [8]:
from lightgbm import LGBMClassifier

lgb_clf = LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    num_leaves=31,
    random_state=42,
    verbose=-1
)

lgb_clf.fit(train_inputs, train_targets)

def evaluate_lgbm(model, inputs, targets, name=''):
    probs = model.predict_proba(inputs)[:, 1]
    auc = roc_auc_score(targets, probs)
    print(f"AUROC {name}: {auc:.4f}")
    return auc

print("Результати:")
train_auc_lgb = evaluate_lgbm(lgb_clf, train_inputs, train_targets, "Train")
val_auc_lgb = evaluate_lgbm(lgb_clf, val_inputs, val_targets, "Val")

Результати:
AUROC Train: 0.9758
AUROC Val: 0.9326


Є перенавчання. Налаштований XGBoos кращий.

5. Використовуючи бібліотеку `Hyperopt` і приклад пошуку гіперпараметрів для `LightGBM` з лекції знайдіть оптимальні значення гіперпараметрів `LightGBM` для нашої задачі. Задайте свою сітку гіперпараметрів виходячи з тих параметрів, які ви б хотіли перебрати. Поставте кількість раундів в підборі гіперпараметрів рівну **10**.

  **Увага!** Для того, аби скористатись hyperopt, нам треба задати функцію `objective`. І тут ми також ставимо loss - негативне значення AUROC, як і при пошуці гіперпараметрів для XGBoost. До речі, можна спробувати написати код так, аби в objective передавати лише модель і не писати схожий код двічі :)

  Після успішного завершення пошуку оптимальних гіперпараметрів
    - виведіть найкращі значення гіперпараметрів
    - створіть в окремій зміній `final_lgb_clf` модель `LightGBM` з найкращими гіперпараметрами
    - навчіть модель `final_lgb_clf`
    - оцініть якість моделі `final_lgb_clf` на тренувальній і валідаційній вибірках з допомогою AUROC.
    - зробіть висновок про якість моделі. Чи стала вона краще порівняно з попереднім пунктом (4) цього завдання?

In [9]:
lgb_space = {
    'n_estimators': hp.quniform('n_estimators', 50, 500, 25),
    'num_leaves': hp.quniform('num_leaves', 20, 150, 1),
    'max_depth': hp.quniform('max_depth', 3, 15, 1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.01), np.log(0.3)),
    'subsample': hp.uniform('subsample', 0.5, 1.0),
    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1.0),
    'min_child_samples': hp.quniform('min_child_samples', 5, 50, 5),
    'reg_alpha': hp.uniform('reg_alpha', 0, 10),
    'reg_lambda': hp.uniform('reg_lambda', 0, 10)
}

def lgb_objective(params):
    params['n_estimators'] = int(params['n_estimators'])
    params['num_leaves'] = int(params['num_leaves'])
    params['max_depth'] = int(params['max_depth'])
    params['min_child_samples'] = int(params['min_child_samples'])

    model = LGBMClassifier(
        **params,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

    model.fit(train_inputs, train_targets)

    probs = model.predict_proba(val_inputs)[:, 1]
    auc = roc_auc_score(val_targets, probs)

    return {'loss': 1 - auc, 'status': STATUS_OK}

lgb_trials = Trials()
lgb_best_params = fmin(
    fn=lgb_objective,
    space=lgb_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=lgb_trials,
    rstate=np.random.default_rng(42)
)

print("\nНайкращі параметри:")
print(lgb_best_params)

100%|██████████| 20/20 [00:12<00:00,  1.57trial/s, best loss: 0.06301838260511694]

Найкращі параметри:
{'colsample_bytree': np.float64(0.6262115027475648), 'learning_rate': np.float64(0.07276908812443281), 'max_depth': np.float64(5.0), 'min_child_samples': np.float64(30.0), 'n_estimators': np.float64(75.0), 'num_leaves': np.float64(112.0), 'reg_alpha': np.float64(5.924534133258074), 'reg_lambda': np.float64(2.1221711699333525), 'subsample': np.float64(0.7018888351309895)}


In [10]:
final_lgb_params = lgb_best_params.copy()
final_lgb_params['n_estimators'] = int(final_lgb_params['n_estimators'])
final_lgb_params['num_leaves'] = int(final_lgb_params['num_leaves'])
final_lgb_params['max_depth'] = int(final_lgb_params['max_depth'])
final_lgb_params['min_child_samples'] = int(final_lgb_params['min_child_samples'])

final_lgb_clf = LGBMClassifier(**final_lgb_params, random_state=42, verbose=-1)
final_lgb_clf.fit(train_inputs, train_targets)

train_auc = roc_auc_score(train_targets, final_lgb_clf.predict_proba(train_inputs)[:, 1])
val_auc = roc_auc_score(val_targets, final_lgb_clf.predict_proba(val_inputs)[:, 1])

print(f"AUROC Train: {train_auc:.4f}")
print(f"AUROC Val:   {val_auc:.4f}")
print(f"Різниця:     {train_auc - val_auc:.4f}")

AUROC Train: 0.9429
AUROC Val:   0.9370
Різниця:     0.0059


Модель стала значно кращою, показує більший результат на валідації.

6. Оберіть модель з експериментів в цьому ДЗ і зробіть новий `submission` на Kaggle та додайте код для цього і скріншот скора на публічному лідерборді.
  
  **Напишіть коментар, чому ви обрали саме цю модель?**

  І я вас вітаю - це останнє завдання з цим набором даних 💪 На цьому етапі корисно проаналізувати, які моделі показали себе найкраще і подумати, чому.

In [11]:
test_raw_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/test.csv')

cols_to_drop = ['id', 'CustomerId', 'Surname']
X_test = test_raw_df.drop(columns=cols_to_drop)

categorical_cols = X_test.select_dtypes(include=['object']).columns.tolist()
for col in categorical_cols:
    X_test[col] = X_test[col].astype('category')

xgb_probs = final_clf.predict_proba(X_test)[:, 1]

submission_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/sample_submission.csv')

submission_df['Exited'] = xgb_probs
submission_df.to_csv('/content/drive/MyDrive/Colab Notebooks/submission_xgb_tuned.csv', index=False)

lgb_probs = final_lgb_clf.predict_proba(X_test)[:, 1]

submission_df_l = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/sample_submission.csv')

submission_df_l['Exited'] = lgb_probs
submission_df_l.to_csv('/content/drive/MyDrive/Colab Notebooks/submission_lgb_tuned.csv', index=False)

обираю дві моделі, оскільки вони показують майже однакові результати: XGBoost (0.9377) та LightGBM (0.9370).